<a href="https://colab.research.google.com/github/Prktriesitout/Langchain_Projects/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

HR AI Assistant for Resume Screening

A mid-sized IT company receives hundreds of resumes daily for open job roles. HR recruiters spend hours manually screening resumes to check if candidates match job descriptions. This process is time-consuming, repetitive, and prone to human bias.

To solve this, the company wants to deploy an AI-powered HR Assistant that can:

Read candidate resumes.
Compare them against job descriptions.
Summarize strengths and highlight relevant skills.
Provide a recommendation (Shortlist/Reject) with a clear justification.
This system will act as the first-level screening tool, helping HR teams reduce workload and focus only on qualified candidates.

As an AI developer, your task is to build an AI Resume Screening Assistant using LangChain Prompt Templates and a HuggingFace LLM (not OpenAI). The assistant should:

Accept resume text and job description as inputs.
Extract and summarize the candidate’s strengths in 3–4 bullet points.
Match candidate skills against the job requirements.
Give a decision (Shortlist / Reject) with one-line reasoning.

Steps Undertaken to Solve this Problem
Step 1: Load a HuggingFace Model / Amzon Bedrock Model

Use HuggingFace’s pipeline with a fine-tuned instruction model (e.g., tiiuae/falcon-7b-instruct).
Wrap it with HuggingFacePipeline so it integrates seamlessly with LangChain.

In [1]:
!pip install -q langchain-huggingface huggingface_hub

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.output_parsers import StrOutputParser



In [3]:
!pip install -q transformers accelerate bitsandbytes langchain-huggingface


loading Hugging Face model

In [4]:
import os
import torch
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

# 1. Load your Hugging Face token from Colab Secrets
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("hugging")
os.environ["HF_TOKEN"] = os.environ["HUGGINGFACEHUB_API_TOKEN"]  # Used by transformers/huggingface_hub

# 2. Load tokenizer and model onto Colab's T4 GPU
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 3. Create pipeline (return_full_text=False prevents echoing the prompt)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
)

# 4. Wrap in LangChain
llm_local = HuggingFacePipeline(pipeline=pipe)
llm = ChatHuggingFace(llm=llm_local)

# 5. Test invocation
print(llm.invoke("Explain RAG in two short sentences.").content)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAG stands for Retrieval-Augmented Generation, which is an approach that combines the strengths of retrieval and generation to improve the quality and relevance of responses to queries.


In [6]:
template = """
You are an AI HR Assistant . Your task is to screen resumes.
### Candidate Resume:
{resume_text}
### Job Description:
{job_description}
### Instruction:
1. Summarize the candidate's strength in 3-4 bullet points.
2. Highlight skills that match the job description.
3. Provide a recommendation: "Shortlist" or "Reject" with a clear justification.
"""

prompt = PromptTemplate.from_template(template)

In [7]:
chain = prompt | llm | StrOutputParser()

In [9]:
resume = """
Vaibhav Pandey, 10 years of experience in travelling , GATE qualified , worked in travelling agency """
job_desc = """
looking for a govt. job in travelling industry where there is a fixed salary """

result = chain.invoke({"resume_text": resume, "job_description": job_desc})
print(result)

### Summary of Candidate's Strengths:

1. **Experienced Travel Industry**: Vaibhav has 10 years of experience working in the travel industry, which aligns well with the job description seeking someone with experience in the same sector.

2. **GATE Qualified**: His qualification from the Graduate Aptitude Test in Engineering (GATE) indicates strong technical and analytical skills, which could be beneficial in a government role.

3. **Fixed Salary Preference**: Vaibhav’s preference for a fixed salary matches the requirement for a government job where salaries are typically set and not negotiable.

### Skills That Match the Job Description:

1. **Travel Industry Experience**: Vaibhav’s extensive experience in the travel industry makes him well


In [12]:
import json, re
from datetime import date
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

EMPLOYEES = {
    "E101": {"name": "Aarav Mehta", "role": "Data Analyst", "dept": "Analytics",
             "manager": "Neha Rao", "join_date": "2026-11-17", "status": "onboarding"},
    "E102": {"name": "Priya Nair", "role": "Backend Engineer", "dept": "Engineering",
             "manager": "Karan Shah", "join_date": "2024-03-04", "status": "active"},
}

TASKS = {
    "E101": [
        {"id": "T1", "phase": "Day 1-30",  "task": "Submit ID proof, PAN and bank details", "done": False},
        {"id": "T2", "phase": "Day 1-30",  "task": "Complete IT setup (laptop, email, VPN)", "done": False},
        {"id": "T3", "phase": "Day 1-30",  "task": "Finish compliance and POSH training",   "done": False},
        {"id": "T4", "phase": "Day 31-60", "task": "Complete first project with mentor",    "done": False},
        {"id": "T5", "phase": "Day 61-90", "task": "90-day review with manager",            "done": False},
    ],
    "E102": [],
}

OFFBOARDING_TEMPLATE = [
    "Return laptop, ID card and access badge",
    "Complete knowledge-transfer handover document",
    "Revoke system and email access (IT)",
    "Exit interview with HR",
    "Full & Final settlement processed (within 45 days of last working day)",
]

@tool
def get_employee_profile(emp_id: str) -> str:
    """Get an employee's profile (name, role, department, manager, join date, status)."""
    return json.dumps(EMPLOYEES.get(emp_id, {"error": "Employee not found"}))

@tool
def get_onboarding_tasks(emp_id: str, pending_only: bool = True) -> str:
    """List onboarding tasks for an employee. Set pending_only=false to see completed ones too."""
    tasks = TASKS.get(emp_id)
    if tasks is None:
        return json.dumps({"error": "Employee not found"})
    if pending_only:
        tasks = [t for t in tasks if not t["done"]]
    return json.dumps(tasks)

@tool
def mark_task_complete(emp_id: str, task_id: str) -> str:
    """Mark one onboarding task as completed, using the employee id and task id (e.g. T1)."""
    for t in TASKS.get(emp_id, []):
        if t["id"] == task_id:
            t["done"] = True
            return json.dumps({"ok": True, "completed": t["task"]})
    return json.dumps({"error": "Task not found"})

@tool
def start_offboarding(emp_id: str, last_working_day: str) -> str:
    """Start offboarding for an employee. last_working_day must be YYYY-MM-DD."""
    if emp_id not in EMPLOYEES:
        return json.dumps({"error": "Employee not found"})
    EMPLOYEES[emp_id]["status"] = "offboarding"
    TASKS[emp_id] = [{"id": f"X{i+1}", "phase": "Offboarding", "task": t, "done": False}
                     for i, t in enumerate(OFFBOARDING_TEMPLATE)]
    return json.dumps({"ok": True, "last_working_day": last_working_day, "tasks": TASKS[emp_id]})

LETTER_TEMPLATES = {
    "welcome_email": "Dear {name},\n\nWelcome to the team! You join as {role} in {dept} on {join_date}. "
                     "Your manager will be {manager}.\n\nRegards,\nHR Team",
    "experience_letter": "TO WHOM IT MAY CONCERN\n\nThis is to certify that {name} worked with us as {role} "
                         "in {dept} from {join_date} to {end_date}. Conduct was satisfactory.\n\nHR Team",
}

@tool
def draft_hr_letter(letter_type: str, emp_id: str, end_date: str = "") -> str:
    """Draft an HR letter from a template. letter_type is 'welcome_email' or 'experience_letter'."""
    emp = EMPLOYEES.get(emp_id)
    if not emp:
        return json.dumps({"error": "Employee not found"})
    tpl = LETTER_TEMPLATES.get(letter_type)
    if not tpl:
        return json.dumps({"error": f"Unknown letter type. Use one of {list(LETTER_TEMPLATES)}"})
    return tpl.format(end_date=end_date or str(date.today()), **emp)

TOOLS = [get_employee_profile, get_onboarding_tasks, mark_task_complete, start_offboarding, draft_hr_letter]
TOOL_MAP = {t.name: t for t in TOOLS}


In [13]:
SYSTEM = """You are an HR Onboarding & Lifecycle Assistant.
- Use the tools for any employee data or action. Never invent task lists or employee details.
- Call a tool only when needed, then answer clearly using the tool results.
- If the user raises harassment/POSH complaints, disciplinary action or legal disputes, do NOT handle it:
  tell them it is being escalated to a human HR representative."""

TOOL_SCHEMAS = [convert_to_openai_tool(t) for t in TOOLS]
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

def run_agent(user_msg, max_steps=5, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_msg}]
    for step in range(max_steps):
        prompt = tokenizer.apply_chat_template(
            messages, tools=TOOL_SCHEMAS, tokenize=False, add_generation_prompt=True
        )
        out = pipe(prompt, max_new_tokens=512)[0]["generated_text"].strip()
        calls = []
        for raw in TOOL_CALL_RE.findall(out):
            try:
                calls.append(json.loads(raw))
            except json.JSONDecodeError:
                pass

        if not calls:                       # no tool call -> final answer
            return out

        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            name, args = c.get("name"), c.get("arguments", {})
            if verbose:
                print(f"🔧 {name}({args})")
            try:
                result = TOOL_MAP[name].invoke(args)
            except Exception as e:
                result = json.dumps({"error": str(e)})
            messages.append({"role": "tool", "name": name, "content": str(result)})
    return "Stopped: too many steps."

In [14]:
print(run_agent("What are the pending onboarding tasks for employee E101?"))
print(run_agent("Aarav (E101) finished IT setup, mark T2 as done and tell me what's left."))
print(run_agent("Draft a welcome email for E101."))
print(run_agent("Priya (E102) is resigning, last working day 2026-12-15. Start offboarding and show the checklist."))
print(run_agent("Draft an experience letter for E102 with end date 2026-12-15."))
print(run_agent("An employee wants to file a harassment complaint against their manager."))  # should escalate

🔧 get_onboarding_tasks({'emp_id': 'E101', 'pending_only': True})
The pending onboarding tasks for employee E101 are:

- Task T1: Submit ID proof, PAN and bank details (Phase: Day 1-30, Not done)
- Task T2: Complete IT setup (laptop, email, VPN) (Phase: Day 1-30, Not done)
- Task T3: Finish compliance and POSH training (Phase: Day 1-30, Not done)
- Task T4: Complete first project with mentor (Phase: Day 31-60, Not done)
- Task T5: 90-day review with manager (Phase: Day 61-90, Not done)

All tasks are currently pending.
🔧 mark_task_complete({'emp_id': 'E101', 'task_id': 'T2'})
🔧 get_onboarding_tasks({'emp_id': 'E101', 'pending_only': False})
The task **T2 - Complete IT setup (laptop, email, VPN)** has been marked as done.

What's left in Aarav's onboarding tasks:
- **T1 - Submit ID proof, PAN and bank details**: Not yet completed.
- **T3 - Finish compliance and POSH training**: Not yet completed.
- **T4 - Complete first project with mentor**: Not yet completed.
- **T5 - 90-day review wit

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔧 start_offboarding({'emp_id': 'E102', 'last_working_day': '2026-12-15'})
🔧 get_onboarding_tasks({'emp_id': 'E102', 'pending_only': False})
The offboarding process for Priya (E102) has been started with the last working day set to December 15, 2026. Here is the checklist:

- Return laptop, ID card and access badge (not done)
- Complete knowledge-transfer handover document (not done)
- Revoke system and email access (IT) (not done)
- Exit interview with HR (not done)
- Full & Final settlement processed (within 45 days of last working day) (not done)

Please mark these tasks as complete once they are finished.
🔧 draft_hr_letter({'letter_type': 'experience_letter', 'emp_id': 'E102', 'end_date': '2026-12-15'})
The draft experience letter for Priya Nair has been successfully created and sent. Here is the content:

TO WHOM IT MAY CONCERN

This is to certify that Priya Nair worked with us as Backend Engineer in Engineering from 2024-03-04 to 2026-12-15. Conduct was satisfactory.

HR Team

Ple

In [15]:
import json, re, itertools, random
from datetime import date, datetime, timedelta
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

USERS = {
    "E101": {"name": "Aarav Mehta", "role": "employee", "manager": "E200"},
    "E102": {"name": "Priya Nair",  "role": "employee", "manager": "E200"},
    "E200": {"name": "Neha Rao",    "role": "manager",  "manager": None},
    "E300": {"name": "Rohit Sen",   "role": "hr",       "manager": None},
}
LEAVE_BALANCE = {
    "E101": {"casual": 8, "sick": 6, "earned": 12},
    "E102": {"casual": 5, "sick": 6, "earned": 10},
    "E200": {"casual": 7, "sick": 6, "earned": 15},
    "E300": {"casual": 7, "sick": 6, "earned": 15},
}
REIMB_LIMITS = {"travel": 10000, "meals": 3000, "internet": 1500, "medical": 15000}
TICKETS = {}                       # ticket_id -> dict (leave + reimbursement)
_counter = itertools.count(1001)

def business_days(start, end):
    d, n = start, 0
    while d <= end:
        if d.weekday() < 5:
            n += 1
        d += timedelta(days=1)
    return n

def make_employee_tools(uid):
    @tool
    def get_leave_balance() -> str:
        """Get the current user's remaining leave balance for casual, sick and earned leave."""
        return json.dumps(LEAVE_BALANCE[uid])

    @tool
    def apply_leave(leave_type: str, start_date: str, end_date: str, reason: str = "") -> str:
        """Apply for leave. leave_type is casual, sick or earned. Dates must be YYYY-MM-DD."""
        leave_type = leave_type.lower().strip()
        if leave_type not in LEAVE_BALANCE[uid]:
            return json.dumps({"error": f"leave_type must be one of {list(LEAVE_BALANCE[uid])}"})
        try:
            s = datetime.strptime(start_date, "%Y-%m-%d").date()
            e = datetime.strptime(end_date, "%Y-%m-%d").date()
        except ValueError:
            return json.dumps({"error": "Dates must be in YYYY-MM-DD format"})
        if e < s:
            return json.dumps({"error": "end_date is before start_date"})
        if s < date.today() and leave_type != "sick":
            return json.dumps({"error": "Cannot apply for past dates (only sick leave can be backdated)"})
        days = business_days(s, e)
        if days == 0:
            return json.dumps({"error": "Selected dates are all weekends"})
        reserved = sum(t["days"] for t in TICKETS.values()
                       if t["kind"] == "leave" and t["emp_id"] == uid
                       and t["status"] == "pending" and t["type"] == leave_type)
        available = LEAVE_BALANCE[uid][leave_type] - reserved
        if days > available:
            return json.dumps({"error": f"Insufficient {leave_type} leave: need {days}, available {available}"})
        tid = f"LV-{next(_counter)}"
        TICKETS[tid] = {"kind": "leave", "emp_id": uid, "type": leave_type, "start": start_date,
                        "end": end_date, "days": days, "reason": reason, "status": "pending", "comment": ""}
        return json.dumps({"ok": True, "ticket_id": tid, "days": days, "status": "pending manager approval"})

    @tool
    def submit_reimbursement(category: str, amount: float, description: str) -> str:
        """Submit a reimbursement claim. category is travel, meals, internet or medical. amount is in INR."""
        category = category.lower().strip()
        if category not in REIMB_LIMITS:
            return json.dumps({"error": f"category must be one of {list(REIMB_LIMITS)}"})
        if amount <= 0:
            return json.dumps({"error": "amount must be positive"})
        if amount > REIMB_LIMITS[category]:
            return json.dumps({"error": f"Exceeds the {category} limit of INR {REIMB_LIMITS[category]}. Needs an HR exception."})
        tid = f"RB-{next(_counter)}"
        TICKETS[tid] = {"kind": "reimbursement", "emp_id": uid, "type": category, "amount": amount,
                        "description": description, "status": "pending", "comment": ""}
        return json.dumps({"ok": True, "ticket_id": tid, "status": "pending manager approval"})

    @tool
    def check_ticket_status(ticket_id: str) -> str:
        """Check the status of one of the current user's leave or reimbursement tickets."""
        t = TICKETS.get(ticket_id)
        if not t or t["emp_id"] != uid:          # same message either way, so no info leak
            return json.dumps({"error": "Ticket not found"})
        return json.dumps(t)

    @tool
    def list_my_tickets() -> str:
        """List all of the current user's leave and reimbursement tickets."""
        return json.dumps({k: v for k, v in TICKETS.items() if v["emp_id"] == uid})

    return [get_leave_balance, apply_leave, submit_reimbursement, check_ticket_status, list_my_tickets]

In [16]:
def llm_generate(system, user, max_new_tokens=600):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return pipe(prompt, max_new_tokens=max_new_tokens)[0]["generated_text"].strip()

FEEDBACK_SYSTEM = """You are an HR performance-review writing assistant for managers.
Rewrite the manager's rough notes into constructive, professional appraisal feedback.
Rules:
- Use only facts in the notes. Do not invent achievements, numbers or incidents.
- Describe behaviour and impact (Situation-Behaviour-Impact), never personality traits.
- Never mention or infer age, gender, religion, caste, health, family status or other protected attributes.
- Do not give a numeric rating, promotion or salary recommendation.
Output exactly these sections:
**Strengths** (2-3 bullets)
**Areas for improvement** (2-3 bullets, each with a concrete suggestion)
**Suggested OKRs for next quarter** (2-3, each with an Objective and measurable Key Results)"""

def rewrite_feedback(rough_notes, employee_name):
    return llm_generate(FEEDBACK_SYSTEM, f"Employee: {employee_name}\nManager's rough notes:\n{rough_notes}")

def make_manager_tools(uid):
    reports = {e for e, u in USERS.items() if u["manager"] == uid}

    @tool
    def get_team_pending_requests() -> str:
        """List pending leave and reimbursement tickets from the manager's direct reports."""
        return json.dumps({k: {**v, "employee": USERS[v["emp_id"]]["name"]} for k, v in TICKETS.items()
                           if v["emp_id"] in reports and v["status"] == "pending"})

    @tool
    def decide_request(ticket_id: str, decision: str, comment: str = "") -> str:
        """Approve or reject a direct report's ticket. decision is 'approved' or 'rejected'."""
        t = TICKETS.get(ticket_id)
        if not t or t["emp_id"] not in reports:
            return json.dumps({"error": "Ticket not found among your direct reports"})
        if decision not in ("approved", "rejected"):
            return json.dumps({"error": "decision must be 'approved' or 'rejected'"})
        if t["status"] != "pending":
            return json.dumps({"error": f"Ticket already {t['status']}"})
        t["status"], t["comment"] = decision, comment
        if decision == "approved" and t["kind"] == "leave":
            LEAVE_BALANCE[t["emp_id"]][t["type"]] -= t["days"]
        return json.dumps({"ok": True, "ticket_id": ticket_id, "status": decision})

    @tool
    def write_appraisal_feedback(employee_id: str, rough_notes: str) -> str:
        """Turn a manager's rough bullet points about a direct report into appraisal feedback and OKR suggestions."""
        if employee_id not in reports:
            return json.dumps({"error": "You can only write feedback for your direct reports"})
        return rewrite_feedback(rough_notes, USERS[employee_id]["name"])

    return [get_team_pending_requests, decide_request, write_appraisal_feedback]

In [17]:
MIN_RESPONSES = 5                 # k-anonymity style threshold
PULSE_RESPONSES = {}              # survey_id -> list of texts (NO employee id stored)
PULSE_PARTICIPANTS = {}           # survey_id -> set of emp_ids (only to stop duplicate submissions)

def scrub_pii(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", text)
    text = re.sub(r"\+?\d[\d\s-]{8,}\d", "[PHONE]", text)
    for u in USERS.values():
        for part in u["name"].split():
            text = re.sub(rf"\b{re.escape(part)}\b", "[NAME]", text, flags=re.I)
    return text

def make_pulse_submit_tool(uid):
    @tool
    def submit_pulse_feedback(survey_id: str, feedback: str) -> str:
        """Submit anonymous feedback for a pulse survey. The response is stored without the user's identity."""
        voters = PULSE_PARTICIPANTS.setdefault(survey_id, set())
        if uid in voters:
            return json.dumps({"error": "You have already responded to this survey"})
        voters.add(uid)
        PULSE_RESPONSES.setdefault(survey_id, []).append(scrub_pii(feedback))
        random.shuffle(PULSE_RESPONSES[survey_id])      # break any submission-order link
        return json.dumps({"ok": True, "message": "Thanks, your response was recorded anonymously"})
    return submit_pulse_feedback

PULSE_SYSTEM = """You summarize anonymous employee pulse-survey responses for HR.
Rules:
- Never quote text that could identify a person, team or incident. Paraphrase only.
- Do not guess who wrote anything.
- Output: **Overall sentiment** (positive/mixed/negative, one line), **Top themes** (3-5 bullets with approximate share
  such as 'about half'), **Suggested actions** (2-3 bullets)."""

def make_pulse_summary_tool():
    @tool
    def summarize_pulse(survey_id: str) -> str:
        """Summarize recurring themes and sentiment of an anonymous pulse survey."""
        responses = PULSE_RESPONSES.get(survey_id, [])
        if len(responses) < MIN_RESPONSES:
            return json.dumps({"error": f"Only {len(responses)} responses. A minimum of {MIN_RESPONSES} is needed to protect anonymity."})
        joined = "\n".join(f"- {scrub_pii(r)}" for r in responses)
        return llm_generate(PULSE_SYSTEM, f"{len(responses)} responses:\n{joined}")
    return summarize_pulse

# Seed data for testing (as if 6 employees had already submitted)
PULSE_RESPONSES["Q4-2026"] = [
    "Workload has been too high since the October release, weekends are getting eaten.",
    "Love my team, but I don't get clear feedback from my manager.",
    "Hybrid policy is great. Would like more learning budget.",
    "Too many meetings, hard to get focused work done.",
    "Career growth path is unclear, no one talks about promotions.",
    "Workload is heavy, and sometimes Priya and I both get the same tasks.",
]

In [18]:
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

def build_tools_for(uid):
    role = USERS[uid]["role"]
    tools = make_employee_tools(uid) + [make_pulse_submit_tool(uid)]
    if role == "manager":
        tools += make_manager_tools(uid) + [make_pulse_summary_tool()]
    if role == "hr":
        tools += [make_pulse_summary_tool()]
    return tools

def run_agent(user_msg, uid, max_steps=5, verbose=True):
    tools = build_tools_for(uid)
    tool_map = {t.name: t for t in tools}
    schemas = [convert_to_openai_tool(t) for t in tools]
    system = f"""You are an HR Self-Service Assistant. Today is {date.today()} ({date.today():%A}).
You are talking to {USERS[uid]['name']} (role: {USERS[uid]['role']}).
- Use tools for every data lookup or action. Never invent balances, tickets or statuses.
- Convert relative dates ('tomorrow', 'next Monday') into YYYY-MM-DD before calling tools.
- If a tool returns an error, explain it to the user in plain words.
- Harassment/POSH complaints, disciplinary action and legal disputes must NOT be handled by you:
  say you are escalating to a human HR representative."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user_msg}]

    for _ in range(max_steps):
        prompt = tokenizer.apply_chat_template(messages, tools=schemas, tokenize=False, add_generation_prompt=True)
        out = pipe(prompt, max_new_tokens=700)[0]["generated_text"].strip()
        calls = []
        for raw in TOOL_CALL_RE.findall(out):
            try:
                calls.append(json.loads(raw))
            except json.JSONDecodeError:
                pass
        if not calls:
            return out
        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            name, args = c.get("name"), c.get("arguments", {})
            if isinstance(args, str):
                args = json.loads(args)
            if verbose:
                print(f"🔧 [{uid}] {name}({args})")
            if name not in tool_map:
                result = json.dumps({"error": "Unknown or unauthorized tool"})
            else:
                try:
                    result = tool_map[name].invoke(args)
                except Exception as e:
                    result = json.dumps({"error": str(e)})
            messages.append({"role": "tool", "name": name, "content": str(result)})
    return "Stopped: too many steps."

In [19]:
tomorrow = date.today() + timedelta(days=1)
mon = date.today() + timedelta(days=(7 - date.today().weekday()))   # next Monday

# --- Leave & reimbursement (Aarav, employee) ---
print(run_agent("How many casual leaves do I have left?", "E101"))
print(run_agent(f"Apply casual leave from {mon} to {mon + timedelta(days=1)} for a family function.", "E101"))
print(run_agent("Apply earned leave for 40 days starting next month.", "E101"))       # should fail: insufficient balance
print(run_agent("I want to claim INR 2,400 for a client lunch, meals category.", "E101"))
print(run_agent("Claim INR 50,000 travel reimbursement.", "E101"))                    # should fail: over limit
print(run_agent("Show my tickets", "E101"))

# --- Manager actions (Neha) ---
print(run_agent("Show pending requests from my team.", "E200"))
print(run_agent("Approve LV-1001 with comment 'Enjoy'.", "E200"))      # use the real id from your run
print(run_agent("What is the status of LV-1001?", "E101"))
print(run_agent("What is the status of LV-1001?", "E102"))             # RBAC: Priya must NOT see it

# --- Performance feedback (Neha) ---
print(run_agent("""Write appraisal feedback for E101. Notes: delivered the churn dashboard 2 weeks early,
good with stakeholders, SQL is strong, misses standups sometimes, docs are messy,
wants to move into ML.""", "E200"))
print(run_agent("Write appraisal feedback for E300", "E200"))          # not a direct report -> error

# --- Pulse survey ---
print(run_agent("Submit anonymous feedback for survey Q4-2026: the on-call rotation is exhausting.", "E101"))
print(run_agent("Summarize pulse survey Q4-2026", "E200"))
print(run_agent("Summarize pulse survey Q4-2026", "E101"))             # employee has no such tool
print(run_agent("Employee wants to report harassment by a colleague.", "E101"))   # escalation

🔧 [E101] get_leave_balance({})
You currently have 8 casual leaves remaining.
🔧 [E101] apply_leave({'leave_type': 'casual', 'start_date': '2026-11-23', 'end_date': '2026-11-24', 'reason': 'family function'})
Your leave application for casual leave from 2026-11-23 to 2026-11-24 has been successfully submitted. The ticket ID is LV-1001. It is currently pending manager approval. Please keep this ticket ID for future reference.
🔧 [E101] apply_leave({'leave_type': 'earned', 'start_date': '2027-01-01', 'end_date': '2027-01-31', 'reason': ''})
It appears that you do not have enough earned leave available for your request. You currently have 12 days of earned leave remaining. Please consider applying for other types of leave or using up some of your existing earned leave.
🔧 [E101] submit_reimbursement({'category': 'meals', 'amount': 2400, 'description': 'Client Lunch'})
Your reimbursement request for INR 2,400 for a client lunch has been submitted successfully. The ticket ID is RB-1002 and it i